In [86]:
import pandas as pd

In [87]:
data = pd.read_csv('data/ingredients.csv', sep=',')
display(data)

,id,name,packing,store,price,type
0,1,сахар,1 кг,лента,62.99,продукты
1,2,печенье на декор,1 упаковка,лента,89.99,декор
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты
3,4,малина и голубика в сердце,1 упаковка,перекресток,579.99,декор
4,5,яйцо с1 окское,10 шт,лента,109.99,продукты
...,...,...,...,...,...,...
134,135,трайфлы,100 шт,ольга,1200.00,упаковка
135,136,коробка кап6,1 шт,ольга,60.00,упаковка
136,137,яйцо с0 365 дней,10 штук,лента,99.99,продукты
137,138,малина с/м,1 кг,ольга,1600.00,продукты


In [88]:
data[data['type'] =='продукты']['packing'].value_counts()

packing
1 кг          18
80 г           8
370 мл         8
300 г          6
1 л            6
2 кг           4
10 штук        4
5 л            3
10 шт          3
500 мл         3
260 мл         2
220 г          2
500 г          2
580 мл         2
850 мл         2
10 г           2
50 г           2
350 г          1
75 г           1
2,2 кг         1
10 кг          1
450 г          1
2 г            1
425 мл         1
100 г          1
150 г          1
180 г          1
1,5 г          1
30 шт          1
0.5 л          1
200 мл         1
1 упаковка     1
90 г           1
Name: count, dtype: int64

>Задача №1: получить столбец с ценой за 100г/100мл ингридиента
Для этого: 
* для упаковки "_ кг" привести к цене за 100 г 
* для упаковки "_ л" привести к цене за 100 мл
* для упаковки "_ штук" привести к цене за 1 штуку
* для упаковки "1 упаковка" оставить как есть
* для упаковки "_ кг" привести к цене за 100 г

In [89]:
def to_gramm(string):
    string = string.split(' ')

    if string[1] == 'кг':
        if ',' in string[0]:
            return float(string[0].replace(',','.')) * 1000
        else:
            return float(string[0]) * 1000
    elif string[1] == 'л':
            if ',' in string[0]:
                return float(string[0].replace(',','.')) * 1000
            else:
                return float(string[0]) * 1000
    else:
        if ',' in string[0]:
            return float(string[0].replace(',','.'))
        else:
            return float(string[0])

In [90]:
to_gramm('1 упаковка')

1.0

In [91]:
def units(string):
    string = string.split(' ')

    if string[1] == 'кг' or string[1] == 'г':
        return 'г'
    elif string[1] == 'л' or string[1] == 'мл':
        return 'мл'
    elif string[1] == 'штук' or string[1] == 'шт' or string[1] == 'лист':
        return 'шт'
    elif string[1] == 'упаковка':
        return 'уп'
    else:
        pass


In [92]:
data['units'] = data['packing'].apply(units)

In [93]:
data['package weight'] = data['packing'].apply(to_gramm)

In [94]:
data['price for unit'] = 0
data.head(3)

,id,name,packing,store,price,type,units,package weight,price for unit
0,1,сахар,1 кг,лента,62.99,продукты,г,1000.0,0
1,2,печенье на декор,1 упаковка,лента,89.99,декор,уп,1.0,0
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты,мл,370.0,0


In [95]:
mask1 = (data['units'] == 'шт') | (data['units'] == 'уп')
mask2 = (data['units'] == 'г') | (data['units'] == 'мл')

In [96]:
data.loc[mask1,'price for unit'] = data.loc[mask1, 'price'] / data.loc[mask1,'package weight']
data.loc[mask2,'price for unit'] = (100 / data.loc[mask2, 'package weight']) * data.loc[mask2,'price']

In [97]:
data = data.drop('packing', axis = 1)

В таблице с ингридиентами  название товаров записаны как в чеке. В работе используется короткое название.
Соответственно необходимо произвести предобработку, чтобы привести название в рабочее короткое название

In [ ]:
def rename(string):
    string = string.split(' ')
    if len(string) == 1:
        return ' '.join(string[:2])
    else:
        return ' '.join(string[:2])
#rename('шоколад белый воздушный 80 г')

'шоколад белый'

In [118]:
data['work_name'] = data['name'].apply(rename)

In [135]:
mask3 = data['type']=='продукты'
mask4 = (data['store'] == 'лента') | (data['store'] == 'ольга')
#mask5 = data['store'] == 'ольга'
table_product = data[mask3 & mask4]

In [136]:
table_product

,id,name,store,price,type,units,package weight,price for unit,work_name
0,1,сахар,лента,62.99,продукты,г,1000.0,6.299000,сахар
2,3,сгущенка рогачев,лента,164.99,продукты,мл,370.0,44.591892,сгущенка рогачев
4,5,яйцо с1 окское,лента,109.99,продукты,шт,10.0,10.999000,яйцо с1
5,6,мука макфа 2 кг,лента,139.99,продукты,г,2000.0,6.999500,мука макфа
6,7,карамель топпинг,лента,99.99,продукты,г,300.0,33.330000,карамель топпинг
...,...,...,...,...,...,...,...,...,...
129,130,грецкий орех,ольга,900.00,продукты,г,1000.0,90.000000,грецкий орех
132,133,шоколад темный вес,ольга,1550.00,продукты,г,1000.0,155.000000,шоколад темный
136,137,яйцо с0 365 дней,лента,99.99,продукты,шт,10.0,9.999000,яйцо с0
137,138,малина с/м,ольга,1600.00,продукты,г,1000.0,160.000000,малина с/м


In [139]:
table_product.sort_values(by = ['work_name','price for unit'],ascending=True).head(60)

,id,name,store,price,type,units,package weight,price for unit,work_name
33,34,ананасы кольцами лента 580 мл,лента,179.99,продукты,мл,580.0,31.032759,ананасы кольцами
90,91,ананасы кусочками лента,лента,169.99,продукты,мл,580.0,29.308621,ананасы кусочками
84,85,ананасы кусочками 365 дней,лента,149.99,продукты,мл,425.0,35.291765,ананасы кусочками
8,9,арахис соленый 150 г,лента,49.99,продукты,г,150.0,33.326667,арахис соленый
7,8,арахис соленый 50 г,лента,29.99,продукты,г,50.0,59.980000,арахис соленый
85,86,ванилин dr.bakers интенсив,лента,13.99,продукты,г,2.0,699.500000,ванилин dr.bakers
49,50,ванилин haas 1.5г,лента,6.99,продукты,г,1.5,466.000000,ванилин haas
119,120,вишня с/м,ольга,580.00,продукты,г,1000.0,58.000000,вишня с/м
21,22,вода питьевая 5 л святой источник,лента,134.99,продукты,мл,5000.0,2.699800,вода питьевая
129,130,грецкий орех,ольга,900.00,продукты,г,1000.0,90.000000,грецкий орех


>Задача №2
* Необходимо создать таблицы с рецептами
* В таблице с ингридиентами определиться по какому критерию будем выбирать продукт среди одинаковых имен
* Соединить таблицу рецепт с таблицей ингридиент.

In [140]:
recept_3_choco = pd.read_csv('data/3 шоколада.csv', sep = ',')

In [149]:
recept_3_choco

,product_name,weight
0,яйцо c0,2
1,сахар,60
2,мука макфа,53
3,разрыхлитель весовой,3
4,какао,15
5,шоколад темный,275
6,шоколад молочный,275
7,шоколад белый,275
8,сливки 33%,825
9,желатин листовой,30


In [146]:
price_for_3choco = recept_3_choco.merge(
    table_product,
    left_on='product_name',
    right_on = 'work_name',
    how='inner'
)

In [147]:
price_for_3choco

,product_name,weight,id,name,store,price,type,units,package weight,price for unit,work_name
0,сахар,60,1,сахар,лента,62.99,продукты,г,1000.0,6.299000,сахар
1,мука макфа,53,6,мука макфа 2 кг,лента,139.99,продукты,г,2000.0,6.999500,мука макфа
2,мука макфа,53,13,мука макфа 1 кг,лента,74.99,продукты,г,1000.0,7.499000,мука макфа
3,разрыхлитель весовой,3,127,разрыхлитель весовой,ольга,540.00,продукты,г,1000.0,54.000000,разрыхлитель весовой
4,какао,15,139,какао,лента,50.00,продукты,г,90.0,55.555556,какао
5,шоколад темный,275,83,шоколад темный особый,лента,129.99,продукты,г,80.0,162.487500,шоколад темный
6,шоколад темный,275,90,шоколад темный щедрая душа,лента,99.99,продукты,г,80.0,124.987500,шоколад темный
7,шоколад темный,275,133,шоколад темный вес,ольга,1550.00,продукты,г,1000.0,155.000000,шоколад темный
8,шоколад молочный,275,14,шоколад молочный alpen gold 80 г,лента,89.99,продукты,г,80.0,112.487500,шоколад молочный
9,шоколад молочный,275,88,шоколад молочный щедрая душа,лента,94.99,продукты,г,80.0,118.737500,шоколад молочный


яйца потерял :(

Вопрос: как выбрать теперь товары. Идея вот какая. 1. самый дешевый вариант торта - выбираем при прочих равных минимальную цену за 100 г. 2. недешвый вариант, но качественный

идея. создать признак - мигалку. если одинаковые значение df['name'] мы ищем минимум среди соответстующих df['price']. создаем в столбце True
далее группируемся и считаем сумму